In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T

STORE_DATA_TO_BRONZE = 1
STORE_DATA_TO_SILVER = 1
STORE_DATA_TO_GOLD = 1

NUM_CUSTOMERS = 1000000      # how many customers to generate
NUM_ORDERS    = 5000000      # how many orders to generate (total)

NUM_CUSTOMERS_DELTA_LAKE_TRANSACTION = 12  # split customer writes across N transactions
NUM_ORDER_DELTA_LAKE_TRANSACTION     = 12  # split order writes across N transactions

countries   = ["USA", "Canada", "Germany", "France", "UK", "Spain", "Italy", "Sweden"]
departments = ["Sales", "Engineering", "HR", "Finance", "Marketing", "Support"]
cities      = ["New York", "San Francisco", "Toronto", "Berlin", "Paris", "London", "Madrid", "Rome", "Stockholm"]
products    = ["Laptop", "Phone", "Headphones", "Monitor", "Keyboard", "Mouse", "Tablet", "Printer"]

countries_col   = F.array(*[F.lit(c) for c in countries])
departments_col = F.array(*[F.lit(d) for d in departments])
cities_col      = F.array(*[F.lit(c) for c in cities])
products_col    = F.array(*[F.lit(p) for p in products])


In [ ]:
def build_customers_df(start_id, end_id):
    """Build a customers DataFrame for ids in [start_id, end_id)."""
    return (
        spark.range(start_id, end_id)
            .withColumn("id", F.col("id").cast("bigint"))
            .withColumn("name", F.concat(F.lit("Customer_"), F.col("id")))
            .withColumn(
                "country",
                F.element_at(countries_col, (F.floor(F.rand() * F.lit(len(countries))) + 1).cast("int"))
            )
            .withColumn(
                "city",
                F.element_at(cities_col, (F.floor(F.rand() * F.lit(len(cities))) + 1).cast("int"))
            )
            .withColumn("email", F.concat(F.lower(F.regexp_replace("name", " ", "")), F.lit("@example.com")))
            .withColumn("phone", F.concat(F.lit("+1-"), (F.floor(F.rand() * F.lit(9000000)) + 1000000).cast("string")))
            .select("id", "name", "country", "city", "email", "phone")
    )


def build_orders_df(start_id, end_id):
    """Build an orders DataFrame for ids in [start_id, end_id), with year/month/day partition cols."""
    return (
        spark.range(start_id, end_id)
            .withColumn("order_id", F.col("id").cast("bigint"))
            .withColumn("customer_id", (F.floor(F.rand() * F.lit(NUM_CUSTOMERS)) + 1).cast("bigint"))
            .withColumn(
                "product",
                F.element_at(products_col, (F.floor(F.rand() * F.lit(len(products))) + 1).cast("int"))
            )
            .withColumn("price", F.round(F.rand() * F.lit(495) + F.lit(5), 2))
            .withColumn("quantity", (F.floor(F.rand() * F.lit(10)) + 1).cast("int"))
            .withColumn("total", F.col("price") * F.col("quantity"))
            .withColumn("order_date", F.expr("date_sub(current_date(), cast(rand() * 365 as int))"))
            .withColumn("year",  F.year("order_date"))
            .withColumn("month", F.month("order_date"))
            .withColumn("day",   F.dayofmonth("order_date"))
            .select("order_id", "customer_id", "product", "price", "quantity",
                    "total", "order_date", "year", "month", "day")
    )


def chunk_ranges(total, n_chunks):
    """Return list of (start_inclusive, end_exclusive) id ranges, 1..total split across n_chunks."""
    base, rem = divmod(total, n_chunks)
    ranges = []
    cursor = 1
    for i in range(n_chunks):
        size = base + (1 if i < rem else 0)
        ranges.append((cursor, cursor + size))
        cursor += size
    return ranges


In [ ]:
import gc

if STORE_DATA_TO_BRONZE == 1:
    # Customers as Parquet under Files/bronze/customers, Hive-partitioned by country.
    # Build one chunk, write it, drop the reference, then loop — the JVM plan for the
    # previous batch is free to GC before the next batch is constructed.
    for start, end in chunk_ranges(NUM_CUSTOMERS, NUM_CUSTOMERS_DELTA_LAKE_TRANSACTION):
        df = build_customers_df(start, end)
        (df.write
            .mode("append")
            .partitionBy("country")
            .parquet("Files/bronze/customers"))
        del df
        spark.catalog.clearCache()
        gc.collect()

    # Orders as Parquet under Files/bronze/orders, Hive-partitioned by year/month/day
    for start, end in chunk_ranges(NUM_ORDERS, NUM_ORDER_DELTA_LAKE_TRANSACTION):
        df = build_orders_df(start, end)
        (df.write
            .mode("append")
            .partitionBy("year", "month", "day")
            .parquet("Files/bronze/orders"))
        del df
        spark.catalog.clearCache()
        gc.collect()


In [ ]:
if STORE_DATA_TO_SILVER == 1:
    # Fresh tables — drop any prior Silver state so the commit history starts from zero
    spark.sql("DROP TABLE IF EXISTS customers")
    spark.sql("DROP TABLE IF EXISTS orders")

    # Customers — N append commits, each visible in DESCRIBE HISTORY customers
    for start, end in chunk_ranges(NUM_CUSTOMERS, NUM_CUSTOMERS_DELTA_LAKE_TRANSACTION):
        df = build_customers_df(start, end)
        (df.write
            .mode("append")
            .format("delta")
            .saveAsTable("customers"))
        del df
        spark.catalog.clearCache()
        gc.collect()

    # Orders — N append commits, partitioned by year/month/day
    for start, end in chunk_ranges(NUM_ORDERS, NUM_ORDER_DELTA_LAKE_TRANSACTION):
        df = build_orders_df(start, end)
        (df.write
            .mode("append")
            .format("delta")
            .partitionBy("year", "month", "day")
            .saveAsTable("orders"))
        del df
        spark.catalog.clearCache()
        gc.collect()


In [ ]:
def _gold_target():
    """Return 'warehouse' or 'lakehouse' based on what is attached to this notebook.

    Fabric exposes the default attached item through notebookutils.runtime.context.
    If the default is a Warehouse, we write via the Spark-to-Warehouse connector.
    Otherwise we fall back to a Delta Table in the default Lakehouse.
    """
    try:
        import notebookutils  # Fabric runtime
        ctx = notebookutils.runtime.context
        default_type = (ctx.get("defaultWorkspaceItemType")
                        or ctx.get("defaultItemType")
                        or "").lower()
        if "warehouse" in default_type:
            return "warehouse"
    except Exception:
        pass
    return "lakehouse"


def _chunk_filter(df, id_col, start, end):
    return df.where((F.col(id_col) >= F.lit(start)) & (F.col(id_col) < F.lit(end)))


if STORE_DATA_TO_GOLD == 1:
    target = _gold_target()
    print(f"Gold target detected: {target}")

    # Source = Silver Delta tables. We read once, then stream chunk-by-chunk into Gold
    # so we never materialise a giant list on the driver. Each chunk is its own action.
    silver_customers = spark.table("customers")
    silver_orders    = spark.table("orders")

    if target == "warehouse":
        # Warehouse: Spark-to-Warehouse connector (synapsesql) appends to the target table.
        # Fresh start — drop any prior Gold tables so chunked appends build from zero.
        spark.sql("DROP TABLE IF EXISTS gold.dbo.gold_customers")
        spark.sql("DROP TABLE IF EXISTS gold.dbo.gold_orders")
        for start, end in chunk_ranges(NUM_CUSTOMERS, NUM_CUSTOMERS_DELTA_LAKE_TRANSACTION):
            chunk = _chunk_filter(silver_customers, "id", start, end)
            chunk.write.mode("append").synapsesql("gold.dbo.gold_customers")
            del chunk
            spark.catalog.clearCache()
            gc.collect()

        for start, end in chunk_ranges(NUM_ORDERS, NUM_ORDER_DELTA_LAKE_TRANSACTION):
            chunk = _chunk_filter(silver_orders, "order_id", start, end)
            chunk.write.mode("append").synapsesql("gold.dbo.gold_orders")
            del chunk
            spark.catalog.clearCache()
            gc.collect()
    else:
        # Lakehouse: Delta table with explicit partitionBy, copied from Silver.
        spark.sql("DROP TABLE IF EXISTS gold_customers")
        spark.sql("DROP TABLE IF EXISTS gold_orders")

        for start, end in chunk_ranges(NUM_CUSTOMERS, NUM_CUSTOMERS_DELTA_LAKE_TRANSACTION):
            chunk = _chunk_filter(silver_customers, "id", start, end)
            (chunk.write
                .mode("append")
                .format("delta")
                .partitionBy("country")
                .saveAsTable("gold_customers"))
            del chunk
            spark.catalog.clearCache()
            gc.collect()

        for start, end in chunk_ranges(NUM_ORDERS, NUM_ORDER_DELTA_LAKE_TRANSACTION):
            chunk = _chunk_filter(silver_orders, "order_id", start, end)
            (chunk.write
                .mode("append")
                .format("delta")
                .partitionBy("year", "month", "day")
                .saveAsTable("gold_orders"))
            del chunk
            spark.catalog.clearCache()
            gc.collect()
